# dataset_6_7_pipeline

This notebook is shared by dataset IDs `6` and `7`.
Both final branches are present here in one place:
- Dataset 6 uses the per-gene CIF reference branch.
- Dataset 7 uses the `4f3t.cif` reference branch.

The last two code cells generate all four published outputs in order:
`raw_data_6_noxform.csv`, `raw_data_6_rescaled.csv`, `raw_data_7_noxform.csv`, and `raw_data_7_rescaled.csv`.


In [ ]:
# Dataset 6/7.1 - Rebuild normalized_rawcounts.csv from 6048D_rawCounts.txt.
# Details:
# - Reads the raw-count file from dataset6_7.
# - Recomputes normalized_rawcounts.csv from scratch.
# - Writes the generated table to working/normalized_rawcounts.csv.

from pathlib import Path
import pandas as pd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
WORKING_DIR = DATASET_DIR / "working"
WORKING_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS_OF_INTEREST = [
    "MR10_unmod_1",
    "MR11_unmod_2",
    "MR12_unmod_3",
    "MR1_NT_1",
    "MR2_NT_2",
    "MR3_NT_3",
    "MR4_Amide3_1",
    "MR5_Amide3_2",
    "MR6_Amide3_3",
    "MR7_GNA7_1",
    "MR8_GNA7_2",
    "MR9_GNA7_3",
]
NT_COLS = ["MR1_NT_1", "MR2_NT_2", "MR3_NT_3"]
UNMOD_COLS = ["MR10_unmod_1", "MR11_unmod_2", "MR12_unmod_3"]
AMIDE_COLS = ["MR4_Amide3_1", "MR5_Amide3_2", "MR6_Amide3_3"]
GNA_COLS = ["MR7_GNA7_1", "MR8_GNA7_2", "MR9_GNA7_3"]

raw_counts = pd.read_csv(DATASET_DIR / "6048D_rawCounts.txt", sep="\t", index_col=0)
counts_only = raw_counts[COLUMNS_OF_INTEREST].copy()

total_counts = counts_only.sum(axis=0)
scaling_factors = total_counts / total_counts.mean()
normalized = counts_only.div(scaling_factors, axis=1)

normalized["mean_nt"] = normalized[NT_COLS].mean(axis=1)
normalized["mean_unmod"] = normalized[UNMOD_COLS].mean(axis=1)
normalized["mean_amide"] = normalized[AMIDE_COLS].mean(axis=1)
normalized["mean_gna"] = normalized[GNA_COLS].mean(axis=1)
normalized.insert(0, "ensembl_id", normalized.index)

normalized.to_csv(WORKING_DIR / "normalized_rawcounts.csv", index=False)
print(normalized.shape)
normalized.head(3)


In [ ]:
# Dataset 6/7.2 - Rebuild the alignment labels and save one working sequence example.
# Details:
# - Reads gene_alignments3.csv from dataset6_7.
# - Recomputes log2FC and off_target in memory from normalized counts.
# - Retrieves one full transcript example and writes sequence_example.json/.fasta into working.

from pathlib import Path
import json
import numpy as np
import pandas as pd
import requests

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
WORKING_DIR = DATASET_DIR / "working"

ENSEMBL_HEADERS = {"Accept": "application/json"}
ENSEMBL_LOOKUP_URL = "https://rest.ensembl.org/lookup/id/{gene_id}"
ENSEMBL_SEQUENCE_URL = "https://rest.ensembl.org/sequence/id/{object_id}"

def fetch_sequence_example(ensembl_id: str) -> dict:
    transcript_id = None
    try:
        lookup_response = requests.get(
            ENSEMBL_LOOKUP_URL.format(gene_id=ensembl_id),
            headers=ENSEMBL_HEADERS,
            params={"expand": 1},
            timeout=30,
        )
        lookup_response.raise_for_status()
        lookup_data = lookup_response.json()
        transcript_id = lookup_data.get("canonical_transcript")
        if not transcript_id:
            transcripts = lookup_data.get("Transcript") or []
            if transcripts:
                transcript_id = transcripts[0].get("id")
        if not transcript_id:
            raise ValueError(f"No transcript available for {ensembl_id}")
        transcript_id = transcript_id.split(".", 1)[0]

        sequence_response = requests.get(
            ENSEMBL_SEQUENCE_URL.format(object_id=transcript_id),
            headers=ENSEMBL_HEADERS,
            params={"type": "cdna"},
            timeout=30,
        )
        sequence_response.raise_for_status()
        sequence_data = sequence_response.json()
        sequence = sequence_data.get("seq")
        if sequence:
            return {
                "ensembl_id": ensembl_id,
                "transcript_id": transcript_id,
                "sequence_length": len(sequence),
                "sequence_prefix": sequence[:40],
                "sequence": sequence,
            }
    except Exception as exc:  # noqa: BLE001
        return {
            "ensembl_id": ensembl_id,
            "transcript_id": transcript_id,
            "retrieval_example_error": str(exc),
        }
    return {}

normalized = pd.read_csv(WORKING_DIR / "normalized_rawcounts.csv")[
    ["ensembl_id", "mean_nt", "mean_unmod", "mean_amide", "mean_gna"]
]
alignment_df = pd.read_csv(DATASET_DIR / "gene_alignments3.csv").drop_duplicates("ensembl_id")
alignment_df = alignment_df.merge(normalized, on="ensembl_id", how="left")

with np.errstate(divide="ignore", invalid="ignore"):
    alignment_df["log2FC_unmod"] = np.log2(alignment_df["mean_unmod"] / alignment_df["mean_nt"])
    alignment_df["log2FC_amide"] = np.log2(alignment_df["mean_amide"] / alignment_df["mean_nt"])
    alignment_df["log2FC_gna"] = np.log2(alignment_df["mean_gna"] / alignment_df["mean_nt"])

dup_counts = alignment_df["target_rna"].value_counts()
alignment_df["off_target"] = alignment_df["target_rna"].map(dup_counts).gt(1).astype(int)
alignment_df.to_csv(WORKING_DIR / "alignment_with_labels.csv", index=False)

candidate_ids = []
seen = set()
for value in alignment_df["ensembl_id"].dropna().astype(str):
    gene_id = value.strip()
    if not gene_id.startswith("ENSG") or gene_id in seen:
        continue
    seen.add(gene_id)
    candidate_ids.append(gene_id)
    if len(candidate_ids) >= 12:
        break

sequence_example = {"retrieval_example_error": "No candidate Ensembl IDs were found."}
for gene_id in candidate_ids:
    sequence_example = fetch_sequence_example(gene_id)
    if sequence_example.get("sequence"):
        break

sequence_json = dict(sequence_example)
json_path = WORKING_DIR / "sequence_example.json"
fasta_path = WORKING_DIR / "sequence_example.fasta"
if sequence_example.get("sequence"):
    fasta_path.write_text(
        f">{sequence_example['ensembl_id']}|{sequence_example.get('transcript_id', 'unknown')}\n{sequence_example['sequence']}\n",
        encoding="utf-8",
    )
    sequence_json["fasta_path"] = str(fasta_path)
elif fasta_path.exists():
    fasta_path.unlink()

json_path.write_text(json.dumps(sequence_json, indent=2), encoding="utf-8")

print({key: value for key, value in sequence_json.items() if key != "sequence"})
alignment_df[["ensembl_id", "target_rna", "log2FC_unmod", "log2FC_amide", "log2FC_gna", "off_target"]].head(3)


In [ ]:
# Dataset 6/7.3 - Build one sample unmod / amide / gna PDB directly from a CIF with PyMOL.
# Details:
# - Uses the published per-gene CIF as the starting point.
# - Writes three sample PDBs into working/ before the minimized GRO workflow.
# - The amide and gna edits come from the original test_alignment notebooks.

from pathlib import Path
import json
import numpy as np

import pymol
try:
    pymol.finish_launching(["pymol", "-cq"])
except Exception:  # noqa: BLE001
    pass
from pymol import cmd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
WORKING_DIR = DATASET_DIR / "working"
SHARED_DATA_DIR = DATA_ROOT / "shared_data"

sample_gene = "ENSG00000000419"
sample_cif = SHARED_DATA_DIR / "RNASeq" / "output_cifs" / f"{sample_gene}.cif"
amide_template = SHARED_DATA_DIR / "RNASeq" / "md0" / "4f3t_v4G2c.pdb"
gna_template = SHARED_DATA_DIR / "RNASeq" / "md0" / "5v2h.cif"

def remove_first_ter_after_chain_b(input_pdb: Path, output_pdb: Path, convert_hetatm: bool = False) -> None:
    chain_b_started = False
    ter_skipped = False

    with input_pdb.open("r", encoding="utf-8") as infile, output_pdb.open("w", encoding="utf-8") as outfile:
        for line in infile:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                chain_id = line[21].strip()
                if convert_hetatm and line.startswith("HETATM"):
                    line = line.replace("HETATM", "ATOM  ", 1)
                if chain_id == "B":
                    chain_b_started = True

            if line.startswith("CONECT") or line.startswith("ANISOU"):
                continue
            if chain_b_started and line.startswith("TER") and not ter_skipped:
                ter_skipped = True
                continue

            outfile.write(line)

def remove_5prime_phosphates(object_name: str) -> None:
    cmd.select("phosphate_atoms", "chain B and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")
    cmd.select("phosphate_atoms", "chain C and resi 1 and (name OP1 or name OP2 or name P)")
    cmd.extract("phosphate_group", "phosphate_atoms")
    cmd.remove("phosphate_atoms")

def build_unmod_sample(cif_file: Path, output_pdb: Path) -> None:
    cmd.reinitialize()
    cmd.load(str(cif_file), "guide_target")
    remove_5prime_phosphates("guide_target")
    cmd.save(str(output_pdb), "guide_target")

def build_amide_sample(cif_file: Path, output_pdb: Path) -> None:
    cmd.reinitialize()
    cmd.load(str(cif_file), "guide_target")
    cmd.load(str(amide_template), "amide_template")

    cmd.select("chainB_resi3_4", "guide_target and chain B and (resi 3 or resi 4)")
    cmd.select("chainR_resi2_3", "amide_template and chain R and (resi 2 or resi 3)")
    cmd.create("new_obj_chainR", "chainR_resi2_3")
    cmd.align("new_obj_chainR", "chainB_resi3_4")
    cmd.select("chainB_atoms", "guide_target and chain B and ((resi 3 and name O3') or (resi 4 and (name OP1 or name OP2 or name P or name C5' or name O5')))")
    cmd.select("keep_new_obj_chainR_atoms", "new_obj_chainR and ((resi 2 and (name C or name O or name C6')) or (resi 3 and (name N or name C5')))")
    cmd.remove("new_obj_chainR and not keep_new_obj_chainR_atoms")
    cmd.remove("chainB_atoms")
    cmd.create("guide_target", "guide_target or new_obj_chainR")
    cmd.select("atom1", "guide_target and chain B and resi 3 and name C3'")
    cmd.select("atom2", "guide_target and chain R and name C6'")
    cmd.bond("atom1", "atom2")
    cmd.select("atom3", "guide_target and chain B and resi 4 and name C4'")
    cmd.select("atom4", "guide_target and chain R and name C5'")
    cmd.bond("atom3", "atom4")

    coord_atom3 = np.array(cmd.get_atom_coords("atom3"))
    coord_atom4 = np.array(cmd.get_atom_coords("atom4"))
    vector3_4 = coord_atom4 - coord_atom3
    vector3_4_normalized = vector3_4 / np.linalg.norm(vector3_4)
    current_distance_3_4 = cmd.get_distance("atom3", "atom4")
    translation_vector_3_4 = vector3_4_normalized * (current_distance_3_4 - 1.6)
    if current_distance_3_4 > 1.6:
        cmd.translate(list(-translation_vector_3_4), "atom4")

    coord_atom1 = np.array(cmd.get_atom_coords("atom1"))
    coord_atom2 = np.array(cmd.get_atom_coords("atom2"))
    vector1_2 = coord_atom2 - coord_atom1
    vector1_2_normalized = vector1_2 / np.linalg.norm(vector1_2)
    current_distance_1_2 = cmd.get_distance("atom1", "atom2")
    translation_vector_1_2 = vector1_2_normalized * (current_distance_1_2 - 1.6)
    if current_distance_1_2 > 1.6:
        cmd.translate(list(-translation_vector_1_2), "atom2")

    cmd.alter("atom2", 'chain="B"')
    cmd.alter("atom2", "resi=3")
    cmd.alter("atom2", 'resn="A"')
    cmd.alter("atom2", 'segi="B"')
    cmd.alter("chain R", "resi=4")
    cmd.alter("chain R", 'resn="G"')
    cmd.alter("chain R", 'segi="B"')
    cmd.alter("chain R", 'chain="B"')
    cmd.alter("chain B and resi 3", "resn='R3'")
    cmd.alter("chain B and resi 4", "resn='R4'")
    cmd.sort()
    cmd.valence("guess", "all")

    remove_5prime_phosphates("guide_target")
    cmd.select("r3_r4", "resi 3+4 and chain B")
    cmd.h_add("r3_r4")
    cmd.sort()

    ter_path = WORKING_DIR / f"{sample_gene}_amide_TER.pdb"
    cmd.save(str(ter_path), "guide_target")
    remove_first_ter_after_chain_b(ter_path, output_pdb)

def build_gna_sample(cif_file: Path, output_pdb: Path) -> None:
    cmd.reinitialize()
    cmd.load(str(cif_file), "guide_target")
    cmd.load(str(gna_template), "gna_template")
    cmd.select("t_methyl", "gna_template and chain A and resi 5 and (name C5M or name H5M1 or name H5M2 or name H5M3)")
    cmd.remove("t_methyl")
    cmd.select("sel_gna", "gna_template and chain A and resi 4-6")

    cmd.create("gna", "sel_gna")
    cmd.select("gna46", "gna and (resi 4 or resi 6)")
    cmd.select("main68", "guide_target and chain B and (resi 6 or resi 8)")
    cmd.align("gna46", "main68")

    cmd.remove("gna and (resi 4 or resi 6)")
    cmd.remove("guide_target and chain B and resi 7")
    cmd.alter("gna", "resi=7")
    cmd.alter("gna", 'segi="B"')
    cmd.alter("gna", 'chain="B"')
    cmd.alter("gna", 'resn="GNAU"')
    cmd.create("guide_target", "guide_target or gna")
    cmd.select("atom1", "guide_target and chain B and resi 6 and name O3'")
    cmd.select("atom2", "guide_target and chain B and resi 7 and name P")
    cmd.select("atom3", "guide_target and chain B and resi 7 and name O2G")
    cmd.select("atom4", "guide_target and chain B and resi 8 and name P")
    cmd.bond("atom1", "atom2")
    cmd.bond("atom3", "atom4")

    cmd.select("proton_site", "guide_target and chain B and resi 7 and name C5")
    cmd.edit("proton_site")
    cmd.attach("H", 1, 1)
    cmd.alter("(elem H and neighbor proton_site)", "name='H01'")
    cmd.unpick()

    coord_atom3 = np.array(cmd.get_atom_coords("atom3"))
    coord_atom4 = np.array(cmd.get_atom_coords("atom4"))
    vector3_4 = coord_atom4 - coord_atom3
    vector3_4_normalized = vector3_4 / np.linalg.norm(vector3_4)
    current_distance_3_4 = cmd.get_distance("atom3", "atom4")
    translation_vector_3_4 = vector3_4_normalized * (current_distance_3_4 - 1.6)
    if current_distance_3_4 > 1.6:
        cmd.select("translate_selection", "guide_target and chain B and resi 8-21")
        cmd.translate(list(-translation_vector_3_4), "translate_selection")

    coord_atom1 = np.array(cmd.get_atom_coords("atom1"))
    coord_atom2 = np.array(cmd.get_atom_coords("atom2"))
    vector1_2 = coord_atom2 - coord_atom1
    vector1_2_normalized = vector1_2 / np.linalg.norm(vector1_2)
    current_distance_1_2 = cmd.get_distance("atom1", "atom2")
    translation_vector_1_2 = vector1_2_normalized * (current_distance_1_2 - 1.6)
    if current_distance_1_2 > 1.6:
        cmd.select("translate_selection_1_6", "guide_target and chain B and resi 7-21")
        cmd.translate(list(-translation_vector_1_2), "translate_selection_1_6")

    remove_5prime_phosphates("guide_target")

    ter_path = WORKING_DIR / f"{sample_gene}_gna_TER.pdb"
    cmd.save(str(ter_path), "guide_target")
    remove_first_ter_after_chain_b(ter_path, output_pdb, convert_hetatm=True)

sample_outputs = {
    "unmod": WORKING_DIR / "sample_unmod_from_cif.pdb",
    "amide": WORKING_DIR / "sample_amide_from_cif.pdb",
    "gna": WORKING_DIR / "sample_gna_from_cif.pdb",
}
build_unmod_sample(sample_cif, sample_outputs["unmod"])
build_amide_sample(sample_cif, sample_outputs["amide"])
build_gna_sample(sample_cif, sample_outputs["gna"])

summary = {
    "sample_gene": sample_gene,
    "sample_cif": str(sample_cif),
    "sample_outputs": {key: str(value) for key, value in sample_outputs.items()},
}
(WORKING_DIR / "sample_pdb_generation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
# Dataset 6/7.4 - Dataset 6 PyMOL extraction from minimized GRO files.
# Details:
# - Uses the per-gene base CIF from shared_data/output_cifs as the reference.
# - Extracts the protein-distance table and the RNA-distance table that were written into rawdata_ref_cif.
# - The whole cell is intentionally commented so the Dataset 7 / 4f3t branch remains the default visible path.
#
# from pathlib import Path
# from multiprocessing import Pool
# import csv
# import gc
# import os
#
# import pymol
# try:
#     pymol.finish_launching(["pymol", "-cq"])
# except Exception:  # noqa: BLE001
#     pass
# from pymol import cmd
#
# DATA_ROOT = Path("/media/volume/sirna-features")
# DATASET_DIR = DATA_ROOT / "dataset6_7"
# SHARED_DATA_DIR = DATA_ROOT / "shared_data"
# OUTPUT_CIFS_DIR = SHARED_DATA_DIR / "RNASeq" / "output_cifs"
# OUTPUT_DIR = SHARED_DATA_DIR / "RNASeq" / "md0" / "rawdata_ref_cif"
# DIRECTORIES = {
#     "unmod": SHARED_DATA_DIR / "unmod" / "step4",
#     "amide": SHARED_DATA_DIR / "amide" / "step4",
#     "gna": SHARED_DATA_DIR / "gna" / "step4",
# }
# PROTEIN_INDICES = list(range(22, 859))
# RNA_INDICES = list(range(0, 42))
#
# def collect_pairs():
#     pairs = []
#     for suffix, directory in DIRECTORIES.items():
#         for comp_file in sorted(directory.glob("*.gro")):
#             pairs.append((suffix, comp_file))
#     return pairs
#
# def process_dataset6_pair(pair):
#     suffix, comp_file = pair
#     gene_id = comp_file.stem
#     ref_file = OUTPUT_CIFS_DIR / f"{gene_id}.cif"
#     if not ref_file.exists():
#         return None
#
#     cmd.reinitialize()
#     cmd.load(str(ref_file), "ref")
#     cmd.load(str(comp_file), "comp")
#     if cmd.count_atoms("ref") == 0 or cmd.count_atoms("comp") == 0:
#         return None
#
#     cmd.remove("solvent")
#     cmd.remove("ref and resn CL")
#     cmd.remove("ref and resn NA+")
#     cmd.remove("ref and hydrogen")
#     cmd.remove("comp and hydrogen")
#     cmd.align("comp", "ref")
#
#     cmd.select("not_C3_ref", "ref and not byres name C3'")
#     cmd.select("not_C3_comp", "comp and not byres name C3'")
#     if cmd.count_atoms("not_C3_ref") == 0 or cmd.count_atoms("not_C3_comp") == 0:
#         return None
#
#     protein_distances = []
#     for resi in PROTEIN_INDICES:
#         protein_distances.append(
#             cmd.distance(
#                 f"dist_{resi}",
#                 f"not_C3_ref and resi {resi} and name C",
#                 f"not_C3_comp and resi {resi} and name C",
#             )
#         )
#
#     cmd.select("only_C3_ref", "ref and (name C3' or name C3G)")
#     cmd.select("only_C3_comp", "comp and (name C3' or name C3G)")
#     distances_chain_a = []
#     distances_chain_b = []
#     for resi in range(1, 22):
#         cmd.select("first_rna_base_ref", f"first (only_C3_ref and resi {resi})")
#         cmd.select("first_rna_base_comp", f"first (only_C3_comp and resi {resi})")
#         if cmd.count_atoms("first_rna_base_ref") > 0 and cmd.count_atoms("first_rna_base_comp") > 0:
#             distances_chain_a.append(cmd.get_distance("first_rna_base_ref", "first_rna_base_comp"))
#
#         cmd.select("first_rna_base_comp_b", f"first (only_C3_comp and resi {resi} and not first_rna_base_comp)")
#         cmd.select("first_rna_base_ref_b", f"first (only_C3_ref and resi {resi} and not first_rna_base_ref)")
#         if cmd.count_atoms("first_rna_base_ref_b") > 0 and cmd.count_atoms("first_rna_base_comp_b") > 0:
#             distances_chain_b.append(cmd.get_distance("first_rna_base_ref_b", "first_rna_base_comp_b"))
#
#     cmd.delete("all")
#     return {
#         "suffix": suffix,
#         "comp_name": gene_id,
#         "protein_distances": protein_distances,
#         "rna_distances": distances_chain_a + distances_chain_b,
#     }
#
# def write_dataset6_tables(results):
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     for suffix in DIRECTORIES:
#         protein_path = OUTPUT_DIR / f"raw_data_{suffix}.csv"
#         rna_path = OUTPUT_DIR / f"raw_data_{suffix}_rna.csv"
#         suffix_rows = [row for row in results if row["suffix"] == suffix]
#
#         with protein_path.open("w", newline="", encoding="utf-8") as csvfile:
#             writer = csv.writer(csvfile)
#             writer.writerow(["comp_name"] + PROTEIN_INDICES)
#             for row in suffix_rows:
#                 writer.writerow([row["comp_name"]] + [round(value, 4) for value in row["protein_distances"]])
#
#         with rna_path.open("w", newline="", encoding="utf-8") as csvfile:
#             writer = csv.writer(csvfile)
#             writer.writerow(["comp_name"] + RNA_INDICES)
#             for row in suffix_rows:
#                 writer.writerow([row["comp_name"]] + [round(value, 4) for value in row["rna_distances"]])
#
# RUN_DATASET6_PYMOL_BATCH = False
# if RUN_DATASET6_PYMOL_BATCH:
#     pairs = collect_pairs()
#     with Pool(processes=8) as pool:
#         results = list(pool.imap(process_dataset6_pair, pairs))
#         gc.collect()
#     results = [row for row in results if row is not None]
#     write_dataset6_tables(results)
#     print(f"Dataset 6 PyMOL extraction wrote {len(results)} rows into {OUTPUT_DIR}")


In [ ]:
# Dataset 6/7.5 - Dataset 7 PyMOL extraction from minimized GRO files.
# Details:
# - Uses 4f3t.cif as the fixed reference.
# - Extracts the protein-distance tables that were written into rawdata_ref_4f3t.
# - This is the active branch immediately before the 7N -> 7R generation cell.

from pathlib import Path
from multiprocessing import Pool
import csv
import gc

import pymol
try:
    pymol.finish_launching(["pymol", "-cq"])
except Exception:  # noqa: BLE001
    pass
from pymol import cmd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
SHARED_DATA_DIR = DATA_ROOT / "shared_data"
OUTPUT_DIR = SHARED_DATA_DIR / "RNASeq" / "md0" / "rawdata_ref_4f3t"
DIRECTORIES = {
    "unmod": SHARED_DATA_DIR / "unmod" / "step4",
    "amide": SHARED_DATA_DIR / "amide" / "step4",
    "gna": SHARED_DATA_DIR / "gna" / "step4",
}
REF_FILE = DATASET_DIR / "4f3t.cif"
PROTEIN_INDICES = list(range(22, 859))

def collect_pairs():
    pairs = []
    for suffix, directory in DIRECTORIES.items():
        for comp_file in sorted(directory.glob("*.gro")):
            pairs.append((suffix, comp_file))
    return pairs

def process_dataset7_pair(pair):
    suffix, comp_file = pair
    cmd.reinitialize()
    cmd.load(str(REF_FILE), "ref")
    cmd.load(str(comp_file), "comp")
    if cmd.count_atoms("ref") == 0 or cmd.count_atoms("comp") == 0:
        return None

    cmd.remove("solvent")
    cmd.remove("ref and resn CL")
    cmd.remove("ref and resn NA+")
    cmd.remove("ref and hydrogen")
    cmd.remove("comp and hydrogen")
    cmd.align("comp", "ref")

    cmd.select("not_C3_ref", "ref and not byres name C3'")
    cmd.select("not_C3_comp", "comp and not byres name C3'")
    if cmd.count_atoms("not_C3_ref") == 0 or cmd.count_atoms("not_C3_comp") == 0:
        return None

    protein_distances = []
    for resi in PROTEIN_INDICES:
        protein_distances.append(
            cmd.distance(
                f"dist_{resi}",
                f"not_C3_ref and resi {resi} and name C",
                f"not_C3_comp and resi {resi} and name C",
            )
        )

    cmd.delete("all")
    return {
        "suffix": suffix,
        "comp_name": comp_file.stem,
        "protein_distances": protein_distances,
    }

def write_dataset7_tables(results):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for suffix in DIRECTORIES:
        protein_path = OUTPUT_DIR / f"raw_data_{suffix}.csv"
        suffix_rows = [row for row in results if row["suffix"] == suffix]
        with protein_path.open("w", newline="", encoding="utf-8") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["comp_name"] + PROTEIN_INDICES)
            for row in suffix_rows:
                writer.writerow([row["comp_name"]] + [round(value, 4) for value in row["protein_distances"]])

RUN_DATASET7_PYMOL_BATCH = False
if RUN_DATASET7_PYMOL_BATCH:
    pairs = collect_pairs()
    with Pool(processes=8) as pool:
        results = list(pool.imap(process_dataset7_pair, pairs))
        gc.collect()
    results = [row for row in results if row is not None]
    write_dataset7_tables(results)
    print(f"Dataset 7 PyMOL extraction wrote {len(results)} rows into {OUTPUT_DIR}")
else:
    print("Dataset 7 PyMOL batch code loaded. Set RUN_DATASET7_PYMOL_BATCH = True to regenerate rawdata_ref_4f3t.")


In [ ]:
# Dataset 6/7.6 - Build Dataset 6 noxform and rescaled outputs from the reference-CIF branch.
# Details:
# - Rebuilds raw_data_6.csv from the rawdata_ref_cif chemistry tables and their paired *_rna tables.
# - Then rebuilds raw_data_6_noxform.csv and raw_data_6_rescaled.csv.
# - Times the full Dataset 6 generation path and writes timing_dataset6.json.

from pathlib import Path
import json
import time
import pandas as pd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
WORKING_DIR = DATASET_DIR / "working"
SHARED_DATA_DIR = DATA_ROOT / "shared_data"
REF_CIF_DIR = SHARED_DATA_DIR / "RNASeq" / "md0" / "rawdata_ref_cif"

dataset6_start = time.perf_counter()
merged_frames = []
for suffix in ("amide", "gna", "unmod"):
    structure_df = pd.read_csv(REF_CIF_DIR / f"raw_data_{suffix}.csv")
    structure_df["comp_name"] = structure_df["comp_name"].astype(str) + f"_{suffix}"

    rna_df = pd.read_csv(REF_CIF_DIR / f"raw_data_{suffix}_rna.csv")
    rna_df["comp_name"] = rna_df["comp_name"].astype(str) + f"_{suffix}"
    rna_df.columns = ["comp_name"] + [str(900 + idx) for idx in range(len(rna_df.columns) - 1)]

    merged_df = structure_df.merge(rna_df, on="comp_name", how="outer")
    merged_frames.append(merged_df)

raw_data_6 = pd.concat(merged_frames, ignore_index=True)
raw_data_6.to_csv(WORKING_DIR / "raw_data_6.csv", index=False)

label_records = []
for row in alignment_df.itertuples(index=False):
    for suffix in ("amide", "gna", "unmod"):
        label_records.append({
            "comp_name": f"{row.ensembl_id}_{suffix}",
            "log2FC": float(getattr(row, f"log2FC_{suffix}")),
        })
label_frame = pd.DataFrame(label_records)

raw_data_6_noxform = raw_data_6.merge(label_frame, on="comp_name", how="left")
bad_minimization_ids = {"ENSG00000202515", "ENSG00000210107", "ENSG00000275708"}
# These entries had a problem during minimization step.
raw_data_6_noxform = raw_data_6_noxform[
    ~raw_data_6_noxform["comp_name"].str.rsplit("_", n=1).str[0].isin(bad_minimization_ids)
].copy()
# Duplicate entries were not removed in these datasets
raw_data_6_noxform = raw_data_6_noxform.sort_values("comp_name").reset_index(drop=True)
raw_data_6_noxform.to_csv(WORKING_DIR / "raw_data_6_noxform.csv", index=False)

working_6 = raw_data_6_noxform.copy()
if "22" in working_6.columns:
    working_6 = working_6.drop(columns=["22"])
# These entries had a problem during minimization step.
working_6 = working_6[
    ~working_6["comp_name"].str.rsplit("_", n=1).str[0].isin(bad_minimization_ids)
].copy()
# Duplicate entries were not removed in these datasets
working_6 = working_6.sort_values("comp_name").reset_index(drop=True)

feature_cols = [column for column in working_6.columns if column not in {"comp_name", "log2FC"}]
min_values = working_6[feature_cols].min()
ranges = working_6[feature_cols].max() - min_values
kept_cols = [column for column in feature_cols if ranges[column] > 0]

raw_data_6_rescaled = pd.DataFrame({"comp_name": working_6["comp_name"].values})
scaled_numeric = ((working_6[kept_cols] - min_values[kept_cols]) / ranges[kept_cols] * 255).round()
raw_data_6_rescaled[kept_cols] = scaled_numeric.astype(int)
raw_data_6_rescaled["log2FC"] = working_6["log2FC"].values
raw_data_6_rescaled.to_csv(WORKING_DIR / "raw_data_6_rescaled.csv", index=False)

dataset6_elapsed = time.perf_counter() - dataset6_start
dataset6_timing = {
    "dataset": 6,
    "elapsed_seconds": dataset6_elapsed,
    "raw_data_6_shape": list(raw_data_6.shape),
    "raw_data_6_noxform_shape": list(raw_data_6_noxform.shape),
    "raw_data_6_rescaled_shape": list(raw_data_6_rescaled.shape),
}
(WORKING_DIR / "timing_dataset6.json").write_text(json.dumps(dataset6_timing, indent=2), encoding="utf-8")

print(json.dumps(dataset6_timing, indent=2))
raw_data_6_rescaled.head(3)


In [ ]:
# Dataset 6/7.7 - Build Dataset 7 noxform and rescaled outputs from the 4f3t reference branch.
# Details:
# - Rebuilds raw_data_7.csv directly from the rawdata_ref_4f3t chemistry tables.
# - Builds raw_data_7_noxform.csv first, then raw_data_7_rescaled.csv.
# - This is the default final target cell: 7N is generated immediately before 7R.

from pathlib import Path
import json
import time
import pandas as pd

DATA_ROOT = Path("/media/volume/sirna-features")
DATASET_DIR = DATA_ROOT / "dataset6_7"
WORKING_DIR = DATASET_DIR / "working"
SHARED_DATA_DIR = DATA_ROOT / "shared_data"
REF_4F3T_DIR = SHARED_DATA_DIR / "RNASeq" / "md0" / "rawdata_ref_4f3t"

dataset7_start = time.perf_counter()
branch_frames = []
for suffix in ("amide", "gna", "unmod"):
    branch_df = pd.read_csv(REF_4F3T_DIR / f"raw_data_{suffix}.csv")
    branch_df["comp_name"] = branch_df["comp_name"].astype(str) + f"_{suffix}"
    branch_frames.append(branch_df)
raw_data_7 = pd.concat(branch_frames, ignore_index=True)
raw_data_7.to_csv(WORKING_DIR / "raw_data_7.csv", index=False)

label_records = []
for row in alignment_df.itertuples(index=False):
    for suffix in ("amide", "gna", "unmod"):
        label_records.append({
            "comp_name": f"{row.ensembl_id}_{suffix}",
            "log2FC": float(getattr(row, f"log2FC_{suffix}")),
        })
label_frame = pd.DataFrame(label_records)

# Build 7N first.
raw_data_7_noxform = raw_data_7.merge(label_frame, on="comp_name", how="left")
# Duplicate entries were not removed in these datasets
raw_data_7_noxform = raw_data_7_noxform.sort_values("comp_name").reset_index(drop=True)
raw_data_7_noxform.to_csv(WORKING_DIR / "raw_data_7_noxform.csv", index=False)

# Then build 7R from 7N.
working_7 = raw_data_7_noxform.copy()
if "22" in working_7.columns:
    working_7 = working_7.drop(columns=["22"])
bad_minimization_ids = {"ENSG00000202515", "ENSG00000210107", "ENSG00000275708"}
# These entries had a problem during minimization step.
working_7 = working_7[
    ~working_7["comp_name"].str.rsplit("_", n=1).str[0].isin(bad_minimization_ids)
].copy()
# Duplicate entries were not removed in these datasets
working_7 = working_7.sort_values("comp_name").reset_index(drop=True)

feature_cols = [column for column in working_7.columns if column not in {"comp_name", "log2FC"}]
min_values = working_7[feature_cols].min()
ranges = working_7[feature_cols].max() - min_values
kept_cols = [column for column in feature_cols if ranges[column] > 0]

raw_data_7_rescaled = pd.DataFrame({"comp_name": working_7["comp_name"].values})
scaled_numeric = ((working_7[kept_cols] - min_values[kept_cols]) / ranges[kept_cols] * 255).round()
raw_data_7_rescaled[kept_cols] = scaled_numeric.astype(int)
raw_data_7_rescaled["log2FC"] = working_7["log2FC"].values
raw_data_7_rescaled.to_csv(WORKING_DIR / "raw_data_7_rescaled.csv", index=False)

dataset7_elapsed = time.perf_counter() - dataset7_start
dataset7_timing = {
    "dataset": 7,
    "elapsed_seconds": dataset7_elapsed,
    "raw_data_7_shape": list(raw_data_7.shape),
    "raw_data_7_noxform_shape": list(raw_data_7_noxform.shape),
    "raw_data_7_rescaled_shape": list(raw_data_7_rescaled.shape),
}
(WORKING_DIR / "timing_dataset7.json").write_text(json.dumps(dataset7_timing, indent=2), encoding="utf-8")

print(json.dumps(dataset7_timing, indent=2))
raw_data_7_rescaled.head(3)


In [ ]:
# Dataset 6/7.8 - Notes for the minimized GRO workflow that produced the published PyMOL inputs.
# Details:
# - The CIF-to-PDB step above writes pdb_with_modification.pdb or the sample modified PDBs.
# - The minimized GRO generation then runs Gromacs on that modified PDB before the PyMOL extraction cells.

from pathlib import Path

DATA_ROOT = Path("/media/volume/sirna-features")
SHARED_DATA_DIR = DATA_ROOT / "shared_data"

gromacs_steps = [
    'gmx pdb2gmx -f pdb_with_modification.pdb -o structure_processed.gro -p topol.top -i posre.itp',
    'gmx editconf -f structure_processed.gro -o structure_box.gro -c -d 1.0 -bt cubic',
    'gmx solvate -cp structure_box.gro -cs spc216.gro -o structure_solv.gro -p topol.top',
    'gmx grompp -f ions.mdp -c structure_solv.gro -p topol.top -o ions.tpr -maxwarn 3',
    'gmx genion -s ions.tpr -o structure_solv_ions.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15',
    'gmx make_ndx -f structure_solv_ions.gro -o index.ndx',
    'gmx grompp -v -f step4.0_minimization.mdp -o step4.0_minimization.tpr -c structure_solv_ions.gro -r structure_solv_ions.gro -p topol.top -n index.ndx -maxwarn 5',
    'gmx mdrun -v -deffnm step4.0_minimization -ntmpi 1',
]

print("Minimized GRO roots:")
print(SHARED_DATA_DIR / "unmod" / "step4")
print(SHARED_DATA_DIR / "amide" / "step4")
print(SHARED_DATA_DIR / "gna" / "step4")
print()
print("Gromacs sequence:")
for step in gromacs_steps:
    print(step)


In [ ]:
# Dataset 6/7.cleanup - Release large objects and reset notebook state.
# Details:
# - Frees common large tables from memory if they were created above.
# - Closes open matplotlib figures.
# - Resets PyMOL state when it was used in this notebook.

import gc

LARGE_NAMES = [
    "raw_counts",
    "counts_only",
    "normalized",
    "alignment_df",
    "merged_frames",
    "branch_frames",
    "working_6",
    "working_7",
    "raw_data_6",
    "raw_data_6_noxform",
    "raw_data_6_rescaled",
    "raw_data_7",
    "raw_data_7_noxform",
    "raw_data_7_rescaled",
    "raw_data_amide",
    "raw_data_gna",
    "raw_data_unmod",
]

for _name in LARGE_NAMES:
    if _name in globals():
        del globals()[_name]

try:
    import matplotlib.pyplot as plt
    plt.close("all")
except Exception:
    pass

try:
    from pymol import cmd
    cmd.delete("all")
    cmd.reinitialize()
except Exception:
    pass

freed = gc.collect()
print("Cleanup complete.", {"gc_objects_collected": freed})
